# Full-data class-balancing experiment: 7 arms x 5 seeds

Pre-registered in `Ujicoba CycleGAN/BALANCING_PLAN.md`. All arms hold the 301 real
train photographs (217 ripe, 84 unripe; DSC_0093 excluded) and the 87 validation
photographs. Added: nothing (`real_only`), k copies of real unripe
(`oversample_rXXX`) or k blind-review-accepted synthetic unripe photographs from
G0X-EMA raw (`cyclegan_rXXX`), with k = 46 / 90 / 133 for ratios 0.6 / 0.8 / 1.0.

yolov8n, 3000 updates for every arm, batch 8, lr 1e-4, seeds 0-4, validation every
250 updates, selected checkpoint = best validation mAP50:95. The 35 runs are split
18 / 17 across the two GPUs of T4 x2. The test split is never opened here.

In [ ]:
# The input mounts are read only and ultralytics writes next to its working
# directory, so the whole code tree is copied into /kaggle/working first.
import os, shutil, subprocess, sys, time
from pathlib import Path

INPUT = Path('/kaggle/input')
print('mounted:', sorted(p.name for p in INPUT.iterdir()) if INPUT.exists() else 'nothing')

def find(*names, marker=None):
    # Kaggle mounts a dataset at /kaggle/input/<slug> or, when several are
    # attached, at /kaggle/input/datasets/<owner>/<slug>, so the slug is searched
    # by name at any depth before falling back to the marker file.
    for name in names:
        for candidate in [INPUT / name, *sorted(INPUT.rglob(name))]:
            if candidate.is_dir() and (marker is None or (candidate / marker).exists()):
                return candidate
    hits = sorted({p.parent for p in INPUT.rglob(marker)}) if marker else []
    if len(hits) == 1: return hits[0]
    raise SystemExit(f'cannot locate {names} (marker {marker}); found {hits}')

SOURCE = find('melon-cyclegan-code', marker='Ujicoba CycleGAN')
DATASET = find('melon-dataset-regrouped-v4', 'melon-final-reviewed-v4', marker='manifest.csv')
ASSETS = find('melon-benchmark-assets', marker='reviewed_masks.json')

WORK = Path('/kaggle/working')
CODE = WORK / 'code'
if CODE.exists(): shutil.rmtree(CODE)
shutil.copytree(SOURCE, CODE)
HERE = CODE / 'Ujicoba CycleGAN'
os.environ['MELON_DATASET'] = str(DATASET)
os.environ['MELON_ASSETS'] = str(ASSETS)
sys.path.insert(0, str(HERE))

import torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('code:', sorted(p.name for p in HERE.iterdir()))
print('dataset:', (DATASET / 'manifest.csv').exists(), 'assets:', (ASSETS / 'reviewed_masks.json').exists())


def run(*command, cwd=None):
    started = time.time()
    result = subprocess.run([sys.executable, *[str(c) for c in command]], cwd=str(cwd or HERE))
    print(f'-> returncode {result.returncode} in {time.time() - started:.1f}s', flush=True)
    if result.returncode != 0: raise SystemExit(result.returncode)

In [ ]:
# The Kaggle image no longer ships ultralytics; the detector needs it. Nothing
# else is installed, so torch and torchvision stay at the image versions.
import importlib.util
if importlib.util.find_spec('ultralytics') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'], check=True)
import importlib.metadata as meta
for package in ('torch', 'torchvision', 'ultralytics'):
    try: print(package, meta.version(package))
    except meta.PackageNotFoundError: print(package, 'NOT INSTALLED')

In [ ]:
import json
# Kaggle flattens a dataset whose upload holds a single top-level folder, so the
# bundle may sit at the dataset root or one level below it.
BUNDLE = find('melon-balance-candidates-g0x', marker='records.json')
if (BUNDLE / 'balance_candidates_g0x' / 'records.json').exists(): BUNDLE = BUNDLE / 'balance_candidates_g0x'
assert (BUNDLE / 'full' / 'images').is_dir(), f'full-frame images missing under {BUNDLE}'
print('bundle:', BUNDLE)
ARMS = WORK / 'balance_arms'
run(HERE / 'tools' / 'build_balance_arms.py', '--bundle', BUNDLE, '--out', ARMS, '--dataset', DATASET)
summary = json.loads((ARMS / 'arms_summary.json').read_text())
print(json.dumps({k: v for k, v in summary.items() if k != 'arms'}, indent=1))
for name, counts in summary['arms'].items(): print(name, counts['total_per_class'], counts['unripe_to_ripe'])

In [ ]:
import torch, subprocess
ALL_ARMS = ['real_only', 'oversample_r060', 'cyclegan_r060', 'oversample_r080', 'cyclegan_r080',
            'oversample_r100', 'cyclegan_r100']
# 35 runs split 18 / 17 across the two GPUs; every call is one (arms, seeds) block.
PLAN = {0: [(ALL_ARMS, [0, 1]), (ALL_ARMS[:4], [2])],
        1: [(ALL_ARMS[4:], [2]), (ALL_ARMS, [3, 4])]}
GROUPS = [block for gpu in PLAN for block in PLAN[gpu]]
n_gpu = torch.cuda.device_count()
print('visible GPUs:', n_gpu, {g: sum(len(a) * len(s) for a, s in b) for g, b in PLAN.items()})
import shlex
def chain(gpu, blocks, first_index):
    # Paths contain a space ('Ujicoba CycleGAN'): every argument is shell-quoted.
    lines = []
    for offset, (arms, seeds) in enumerate(blocks):
        out = WORK / f'balance_group{first_index + offset}'
        argv = [sys.executable, str(HERE / 'tools' / 'train_detector_arms.py'), '--arms-dir', str(ARMS),
                '--arms', *arms, '--seeds', *map(str, seeds), '--out', str(out),
                '--updates', '3000', '--eval-every', '250', '--export']
        env = f"CUDA_VISIBLE_DEVICES={gpu if n_gpu >= 2 else 0} PYTHONPATH={shlex.quote(str(HERE))}"
        lines.append(env + ' ' + shlex.join(argv))
    log = open(WORK / f'balance_gpu{gpu}.log', 'w')
    return subprocess.Popen(' && '.join(lines), shell=True, cwd=str(HERE), stdout=log, stderr=subprocess.STDOUT)
started = time.time()
if n_gpu >= 2:
    codes = [p.wait() for p in [chain(0, PLAN[0], 0), chain(1, PLAN[1], len(PLAN[0]))]]
else:
    codes = [chain(0, PLAN[0], 0).wait(), chain(0, PLAN[1], len(PLAN[0])).wait()]
print('return codes', codes, 'wall minutes', round((time.time() - started) / 60, 1))
for g in PLAN: print((WORK / f'balance_gpu{g}.log').read_text()[-2000:])
if any(codes): raise SystemExit('detector training failed')

In [ ]:
import statistics
OUT = WORK / 'balance_detector'
OUT.mkdir(exist_ok=True)
results, selected = {}, {}
for i in range(len(GROUPS)):
    part = json.loads((WORK / f'balance_group{i}' / 'results.json').read_text())
    results.update(part['results']); selected.update(part['selected'])
    shutil.copytree(WORK / f'balance_group{i}', OUT / f'group{i}', dirs_exist_ok=True)
arms, seeds = ALL_ARMS, [0, 1, 2, 3, 4]
missing = [f'{a}@{s}' for a in arms for s in seeds if f'{a}@{s}' not in selected]
if missing: raise SystemExit(f'missing runs: {missing}')
def rows_for(table):
    rows = []
    for arm in arms:
        keys = [f'{arm}@{s}' for s in seeds]
        values = [table[k]['map50_95'] for k in keys]
        rows.append(dict(arm=arm, runs=len(values), map50_95=round(statistics.fmean(values), 6),
                         sd=round(statistics.pstdev(values), 6),
                         ap_unripe=round(statistics.fmean(table[k]['per_class']['unripe']['ap50_95'] for k in keys), 6),
                         recall_unripe=round(statistics.fmean(table[k]['recall'][1] for k in keys), 6)))
    return rows
report = dict(results_final=results, results_selected=selected, arms=arms, seeds=seeds, updates=3000,
              eval_every=250, summary_final=rows_for(results), summary_selected=rows_for(selected),
              arms_summary=summary, test_used=False)
(OUT / 'results.json').write_text(json.dumps(report, indent=2) + '\n')
for row in report['summary_selected']: print('selected', row)
for row in report['summary_final']: print('final', row)

In [ ]:
shutil.rmtree(CODE, ignore_errors=True)
shutil.rmtree(ARMS, ignore_errors=True)
for i in range(len(GROUPS)): shutil.rmtree(WORK / f'balance_group{i}', ignore_errors=True)
print(sorted(p.name for p in WORK.iterdir()))